In [1]:
from pyspark.sql import SparkSession

# Question 1.
Execute spark.version.


In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"Spark version: {spark.version}")


26/03/09 22:05:45 WARN Utils: Your hostname, iseohuiui-MacBookAir.local resolves to a loopback address: 127.0.0.1; using 192.168.219.105 instead (on interface en0)
26/03/09 22:05:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 22:05:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 22:05:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.5.1


# Data Load

In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 22:06:11--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net) 해석 중... 52.84.167.55, 52.84.167.175, 52.84.167.134, ...
다음으로 연결 중: d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.84.167.55|:443... 연결했습니다.
HTTP 요청을 보냈습니다. 응답 기다리는 중... 200 OK
길이: 71134255 (68M) [binary/octet-stream]
저장 위치: `yellow_tripdata_2025-11.parquet'

yellow_tripdata_202 100%[===================>]  67.84M  11.7MB/s    /  8.0s    

2026-03-09 22:06:19 (8.50 MB/s) - `yellow_tripdata_2025-11.parquet' 저장함 [71134255/71134255]



In [5]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

# Question 2
Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [7]:
file_path = 'data/pq/nov_2025'

In [11]:
df.repartition(4)\
    .write.parquet(file_path)

# Question 3
How many taxi trips were there on the 15th of November?



In [18]:
from pyspark.sql.functions import dayofmonth

In [21]:
df_3 = df.withColumn("tpep_pickup_day", dayofmonth("tpep_pickup_datetime"))
df_3.filter("tpep_pickup_day = 15").count()

162604

# Question 4
What is the length of the longest trip in the dataset in hours?



In [49]:
from pyspark.sql.functions import row_number, col, desc
from pyspark.sql.window import Window

In [59]:
df_3\
    .withColumn("duration", (col("tpep_dropoff_datetime") - col("tpep_pickup_datetime"))/3600)\
    .withColumn("rownum", row_number().over(Window.partitionBy().orderBy(desc("duration"))))\
    .filter("rownum == 1")\
    .select("tpep_dropoff_datetime", "tpep_pickup_datetime", "duration")\
    .show(truncate=False)

26/03/09 22:31:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 22:31:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/09 22:31:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------------+--------------------+------------------------------------------+
|tpep_dropoff_datetime|tpep_pickup_datetime|duration                                  |
+---------------------+--------------------+------------------------------------------+
|2025-11-30 15:01:00  |2025-11-26 20:22:12 |INTERVAL '0 00:01:30.646667' DAY TO SECOND|
+---------------------+--------------------+------------------------------------------+



# Question 5
Spark's User Interface which shows the application's dashboard runs on which local port?

In [60]:
spark

# Question 6
Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?



In [61]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 22:33:50--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net) 해석 중... 52.84.167.55, 52.84.167.2, 52.84.167.134, ...
다음으로 연결 중: d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.84.167.55|:443... 연결했습니다.
HTTP 요청을 보냈습니다. 응답 기다리는 중... 200 OK
길이: 12331 (12K) [text/csv]
저장 위치: `taxi_zone_lookup.csv'

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    /  0s      

2026-03-09 22:33:50 (1.91 GB/s) - `taxi_zone_lookup.csv' 저장함 [12331/12331]



In [62]:
df_zone = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("taxi_zone_lookup.csv")

In [68]:
df_zone.createOrReplaceTempView("taxi_zone_lookup")

In [67]:
df_3.createOrReplaceTempView("taxi_data")

In [71]:
spark.sql("""
        SELECT *
        FROM (
            SELECT Zone, COUNT(1) AS cnt
            FROM taxi_data td LEFT JOIN taxi_zone_lookup tzl ON td.PULocationID = tzl.LocationID
            GROUP BY Zone
        )
        ORDER BY cnt
""").show(truncate=False)

+---------------------------------------------+---+
|Zone                                         |cnt|
+---------------------------------------------+---+
|Governor's Island/Ellis Island/Liberty Island|1  |
|Eltingville/Annadale/Prince's Bay            |1  |
|Arden Heights                                |1  |
|Port Richmond                                |3  |
|Rikers Island                                |4  |
|Rossville/Woodrow                            |4  |
|Green-Wood Cemetery                          |4  |
|Great Kills                                  |4  |
|Jamaica Bay                                  |5  |
|Westerleigh                                  |12 |
|Crotona Park                                 |14 |
|Oakwood                                      |14 |
|New Dorp/Midland Beach                       |14 |
|West Brighton                                |14 |
|Willets Point                                |15 |
|Breezy Point/Fort Tilden/Riis Beach          |16 |
|Saint Georg